In [1]:
import pandas as pd

survey_25 = pd.read_csv(r'C:\Users\sahil\projects\boston-mbta-ridership\data\processed\ps_2025.csv',
                        dtype={'reporting_group': str})
survey_17 = pd.read_csv(r'C:\Users\sahil\projects\boston-mbta-ridership\data\processed\ps_2017_2019_values.csv',
                        dtype={'line': str})

print(survey_25.shape, survey_17.shape)

(5329, 9) (137, 108)


In [2]:
print('All Bus' in survey_25['reporting_group'].unique())

True


In [3]:
bus17 = survey_17[survey_17['mode'] == 'Bus'].copy()
print(bus17.shape)

(137, 108)


In [4]:
# 2015-17 column -> shared answer name
measures_17 = {
    'trip_purpose': {'trip_purpose_home_based_work':            'work',
                     'trip_purpose_home_based_school':          'school',
                     'trip_purpose_home_based_social_activity': 'social',
                     'trip_purpose_home_based_other':           'other errands',
                     'trip_purpose_non_home_based':             'not from home'},
    'low_income':   {'low_income_classification_yes': 'yes',
                     'low_income_classification_no':  'no'},
    'minority':     {'minority_classification_yes': 'yes',
                     'minority_classification_no':  'no'},
    'vehicles':     {'usable_household_vehicles_0':         '0',
                     'usable_household_vehicles_1':         '1',
                     'usable_household_vehicles_2':         '2',
                     'usable_household_vehicles_3_or_more': '3+'},
}

# Column with the number of survey responses for each question
count_cols_17 = {'trip_purpose': 'count_trip_purpose',
                 'low_income':   'count_low_income_classification',
                 'minority':     'count_minority_classification',
                 'vehicles':     'count_usable_household_vehicles'}

In [5]:
number_cols = [c for cols in measures_17.values() for c in cols] + list(count_cols_17.values())

all_bus = bus17[number_cols].sum().to_frame().T
all_bus['line'] = 'All Bus'

bus17 = pd.concat([bus17[['line'] + number_cols], all_bus], ignore_index=True)
print(bus17.shape)
bus17.tail(2)

(138, 18)


,line,trip_purpose_home_based_work,trip_purpose_home_based_school,trip_purpose_home_based_social_activity,trip_purpose_home_based_other,trip_purpose_non_home_based,low_income_classification_yes,low_income_classification_no,minority_classification_yes,minority_classification_no,usable_household_vehicles_0,usable_household_vehicles_1,usable_household_vehicles_2,usable_household_vehicles_3_or_more,count_trip_purpose,count_low_income_classification,count_minority_classification,count_usable_household_vehicles
136,CT3,845.0,17.0,0.0,138.0,52.0,207.0,621.0,483.0,604.0,242.0,518.0,293.0,121.0,122.0,96.0,126.0,136.0
137,All Bus,220722.0,25878.0,13004.0,38102.0,18739.0,117443.0,166176.0,149270.0,160923.0,134764.0,140139.0,55622.0,15277.0,33584.0,28908.0,31814.0,35596.0


In [6]:
pieces = []

for measure, columns in measures_17.items():
    # Wide -> long: one row per route and answer
    long = bus17.melt(id_vars='line', value_vars=list(columns.keys()),
                      var_name='column', value_name='riders')
    long['category'] = long['column'].map(columns)

    # Share = this answer's riders / all riders answering the question, on that route
    long['pct'] = long['riders'] / long.groupby('line')['riders'].transform('sum') * 100

    long['n_responses'] = long['line'].map(bus17.set_index('line')[count_cols_17[measure]])
    long['measure'] = measure
    pieces.append(long)

s17 = pd.concat(pieces, ignore_index=True)
s17 = s17.rename(columns={'line': 'route_group'})
s17['survey_year'] = '2015-17'
s17 = s17[['survey_year', 'route_group', 'measure', 'category', 'pct', 'n_responses']]

print(s17.shape)
s17[s17['route_group'] == '23']

(1794, 6)


,survey_year,route_group,measure,category,pct,n_responses
13,2015-17,23,trip_purpose,work,59.496338,316.0
151,2015-17,23,trip_purpose,school,6.962978,316.0
289,2015-17,23,trip_purpose,social,7.595064,316.0
427,2015-17,23,trip_purpose,other errands,18.982643,316.0
565,2015-17,23,trip_purpose,not from home,6.962978,316.0
703,2015-17,23,low_income,yes,58.954341,268.0
841,2015-17,23,low_income,no,41.045659,268.0
979,2015-17,23,minority,yes,84.906789,318.0
1117,2015-17,23,minority,no,15.093211,318.0
1255,2015-17,23,vehicles,0,47.125809,348.0


In [7]:
measure_names_25 = {'Trip Purpose':        'trip_purpose',
                    'Title VI Low-Income': 'low_income',
                    'Title VI Minority':   'minority',
                    'Usable Cars':         'vehicles'}

category_names_25 = {'Home-based Work': 'work', 'Home-based School': 'school',
                     'Home-based Social Activity': 'social', 'Home-based Other': 'other errands',
                     'Non-home Based': 'not from home',
                     'Yes': 'yes', 'No': 'no',
                     '0': '0', '1': '1', '2': '2', '3 or more': '3+'}

s25 = survey_25[(survey_25['service_mode'] == 'Bus') &
                (survey_25['measure'].isin(measure_names_25.keys()))].copy()

s25['measure']  = s25['measure'].map(measure_names_25)
s25['category'] = s25['category'].map(category_names_25)
s25 = s25.dropna(subset=['category'])                      # drops "Did not answer"

# Rescale so answers add to 100%, matching 2015-17
s25['pct'] = s25['weighted_percent'] / s25.groupby(['reporting_group', 'measure'])['weighted_percent'].transform('sum') * 100

s25 = s25.rename(columns={'reporting_group': 'route_group', 'n': 'n_responses'})
s25['survey_year'] = '2025'
s25 = s25[['survey_year', 'route_group', 'measure', 'category', 'pct', 'n_responses']]

print(s25.shape)
s25[s25['route_group'] == '23']

(585, 6)


,survey_year,route_group,measure,category,pct,n_responses
1325,2025,23,minority,yes,86.886065,1370
1328,2025,23,minority,no,13.113935,1370
1398,2025,23,low_income,yes,83.948073,1370
1399,2025,23,low_income,no,16.051927,1370
1442,2025,23,trip_purpose,work,39.039851,1370
1447,2025,23,trip_purpose,school,14.713499,1370
1452,2025,23,trip_purpose,social,17.546103,1370
1457,2025,23,trip_purpose,other errands,20.467861,1370
1477,2025,23,trip_purpose,not from home,8.232686,1370
1504,2025,23,vehicles,0,57.636799,1370


In [8]:
surveys = pd.concat([s17, s25], ignore_index=True)
surveys['pct'] = surveys['pct'].round(1)
print(surveys.shape)

surveys.to_csv(r'C:\Users\sahil\projects\boston-mbta-ridership\data\processed\surveys_clean.csv', index=False)

(2379, 6)


In [9]:
check = surveys[(surveys['measure'] == 'low_income') & (surveys['category'] == 'yes') &
                (surveys['route_group'].isin(['23', '28', '22 and 29', '28 & 29', 'All Bus']))]
check.pivot(index='route_group', columns='survey_year', values='pct')

survey_year,2015-17,2025
route_group,,
22 and 29,69.7,NaN
23,59.0,83.9
28,65.0,NaN
28 & 29,NaN,74.6
All Bus,41.4,77.6
